[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_stylized_facts/SFM_ch2_stylized_facts.ipynb)

# SFM_ch2_stylized_facts

Stylized Facts of Financial Return Distributions
Description:
- Fat tails: excess kurtosis and tail probability comparison
- Skewness: negative skewness in equity returns
- Volatility clustering: ACF of squared and absolute returns
- Gain/loss asymmetry: conditional distributions
- Aggregational Gaussianity: daily vs weekly vs monthly
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from statsmodels.tsa.stattools import acf
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean:     {log_ret.mean():.6f}")
print(f"   Std Dev:  {log_ret.std():.6f}")
print(f"   Skewness: {log_ret.skew():.4f}")
print(f"   Kurtosis: {log_ret.kurtosis():.4f}  (excess, Normal = 0)")

## Fat Tails

In [ ]:
# Standardize returns
z = (log_ret - log_ret.mean()) / log_ret.std()

fig, ax = plt.subplots(figsize=(7, 3.5))

# Histogram on log scale
bins = np.linspace(-6, 6, 80)
ax.hist(z, bins=bins, density=True, alpha=0.5, color=MAIN_BLUE,
        edgecolor='white', linewidth=0.3, label='S&P 500 log-returns')

# Normal overlay
x = np.linspace(-6, 6, 300)
ax.plot(x, stats.norm.pdf(x), color=CRIMSON, linewidth=1.2,
        linestyle='--', label='Normal density')

ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(-6, 6)
ax.set_xlabel('Standardized return')
ax.set_ylabel('Density (log scale)')

# Annotate kurtosis
excess_kurt = log_ret.kurtosis()
ax.text(0.97, 0.95, f'Excess kurtosis = {excess_kurt:.2f}\n(Normal = 0)',
        transform=ax.transAxes, ha='right', va='top', fontsize=7,
        color=MAIN_BLUE, bbox=dict(boxstyle='round,pad=0.3',
        facecolor='white', edgecolor=MAIN_BLUE, alpha=0.7, linewidth=0.5))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_fat_tails')

# Tail probability comparison
print('\n   Tail probability comparison (P(|Z| > threshold)):')
print(f"   {'':<10} {'Empirical':>12} {'Normal':>12} {'Ratio':>10}")
for sigma in [3, 4, 5]:
    emp = (np.abs(z) > sigma).mean()
    norm = 2 * stats.norm.sf(sigma)
    ratio = emp / norm if norm > 0 else np.inf
    print(f"   {sigma} sigma   {emp:>12.6f} {norm:>12.6f} {ratio:>10.1f}x")

## Skewness

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

# Panel 1: Conditional densities of positive vs negative returns
pos_ret = log_ret[log_ret > 0]
neg_ret = log_ret[log_ret < 0]

x_pos = np.linspace(0, pos_ret.max(), 200)
x_neg = np.linspace(neg_ret.min(), 0, 200)

kde_pos = stats.gaussian_kde(pos_ret)
kde_neg = stats.gaussian_kde(np.abs(neg_ret))

x_abs = np.linspace(0, max(pos_ret.max(), np.abs(neg_ret.min())), 200)
ax1.plot(x_abs, kde_pos(x_abs), color=FOREST, linewidth=1.2, label='Gains')
ax1.plot(x_abs, kde_neg(x_abs), color=CRIMSON, linewidth=1.2, label='|Losses|')
ax1.fill_between(x_abs, kde_pos(x_abs), kde_neg(x_abs),
                 where=kde_neg(x_abs) > kde_pos(x_abs),
                 alpha=0.15, color=CRIMSON)

ax1.set_xlabel('Absolute return magnitude')
ax1.set_ylabel('Density')
ax1.set_title('Conditional densities: gains vs |losses|', fontweight='bold', fontsize=9)
ax1.legend(loc='upper right', frameon=False, fontsize=7)
ax1.set_xlim(0, 0.06)

# Panel 2: Rolling skewness
rolling_skew = log_ret.rolling(window=252).skew().dropna()

ax2.plot(rolling_skew.index, rolling_skew.values, color=MAIN_BLUE, linewidth=0.7)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax2.fill_between(rolling_skew.index, 0, rolling_skew.values,
                 where=rolling_skew.values < 0, alpha=0.15, color=CRIMSON)
ax2.fill_between(rolling_skew.index, 0, rolling_skew.values,
                 where=rolling_skew.values > 0, alpha=0.15, color=FOREST)

ax2.set_xlabel('Date')
ax2.set_ylabel('Skewness')
ax2.set_title(f'Rolling 1-year skewness (overall = {log_ret.skew():.3f})',
              fontweight='bold', fontsize=9)

ax2.text(0.97, 0.05, 'Predominantly negative',
         transform=ax2.transAxes, ha='right', va='bottom',
         fontsize=7, color=CRIMSON)

plt.tight_layout()
save_fig('ch2_skewness')

## Volatility Clustering

In [ ]:
n_lags = 50
acf_ret = acf(log_ret, nlags=n_lags, fft=True)
acf_sq = acf(log_ret ** 2, nlags=n_lags, fft=True)
acf_abs = acf(np.abs(log_ret), nlags=n_lags, fft=True)

# Confidence band (approximate 95%)
conf = 1.96 / np.sqrt(len(log_ret))
lags = np.arange(1, n_lags + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))

# Panel 1: ACF of r_t
ax1.bar(lags, acf_ret[1:], width=0.6, color=MAIN_BLUE, alpha=0.8)
ax1.axhline(y=conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax1.axhline(y=-conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax1.axhline(y=0, color='gray', linewidth=0.4)
ax1.set_xlabel('Lag $k$')
ax1.set_ylabel('ACF')
ax1.set_title('ACF of $r_t$\n(near zero)', fontweight='bold', fontsize=9)
ax1.set_ylim(-0.08, 0.08)

# Panel 2: ACF of r_t^2
ax2.bar(lags, acf_sq[1:], width=0.6, color=AMBER, alpha=0.8)
ax2.axhline(y=conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax2.axhline(y=-conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax2.axhline(y=0, color='gray', linewidth=0.4)
ax2.set_xlabel('Lag $k$')
ax2.set_ylabel('ACF')
ax2.set_title('ACF of $r_t^2$\n(slow decay)', fontweight='bold', fontsize=9)
ax2.set_ylim(-0.03, max(acf_sq[1:]) * 1.15)

# Panel 3: ACF of |r_t|
ax3.bar(lags, acf_abs[1:], width=0.6, color=FOREST, alpha=0.8)
ax3.axhline(y=conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax3.axhline(y=-conf, color=CRIMSON, linestyle='--', linewidth=0.6)
ax3.axhline(y=0, color='gray', linewidth=0.4)
ax3.set_xlabel('Lag $k$')
ax3.set_ylabel('ACF')
ax3.set_title('ACF of $|r_t|$\n(slow decay)', fontweight='bold', fontsize=9)
ax3.set_ylim(-0.03, max(acf_abs[1:]) * 1.15)

# Remove extra spines on panels 2 and 3
for ax in [ax2, ax3]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig('ch2_vol_clustering')

print(f'\n   ACF(1) of r_t:     {acf_ret[1]:.4f}')
print(f'   ACF(1) of r_t^2:   {acf_sq[1]:.4f}')
print(f'   ACF(1) of |r_t|:   {acf_abs[1]:.4f}')

## Gain/Loss Asymmetry

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

gains = log_ret[log_ret > 0]
losses = log_ret[log_ret < 0]

# Histogram of gains
bins_gain = np.linspace(0, 0.12, 60)
ax.hist(gains, bins=bins_gain, density=True, alpha=0.4, color=FOREST,
        edgecolor='white', linewidth=0.3, label='Gains')

# Histogram of |losses|
ax.hist(np.abs(losses), bins=bins_gain, density=True, alpha=0.4, color=CRIMSON,
        edgecolor='white', linewidth=0.3, label='|Losses|')

# KDE overlays
x_kde = np.linspace(0.0001, 0.12, 300)
kde_gains = stats.gaussian_kde(gains)
kde_losses = stats.gaussian_kde(np.abs(losses))

ax.plot(x_kde, kde_gains(x_kde), color=FOREST, linewidth=1.2)
ax.plot(x_kde, kde_losses(x_kde), color=CRIMSON, linewidth=1.2)

# Shade the region where losses exceed gains (heavier right tail of losses)
ax.fill_between(x_kde, kde_gains(x_kde), kde_losses(x_kde),
                where=kde_losses(x_kde) > kde_gains(x_kde),
                alpha=0.2, color=CRIMSON, label='Losses > Gains')

ax.set_xlabel('Absolute return magnitude')
ax.set_ylabel('Density')

# Annotate key statistics
ax.text(0.97, 0.95,
        f'Mean gain:   {gains.mean():.5f}\n'
        f'Mean |loss|: {np.abs(losses).mean():.5f}\n'
        f'Max gain:    {gains.max():.4f}\n'
        f'Max |loss|:  {np.abs(losses).min():.4f}  / {np.abs(losses).max():.4f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=7,
        color=MAIN_BLUE, bbox=dict(boxstyle='round,pad=0.3',
        facecolor='white', edgecolor=MAIN_BLUE, alpha=0.7, linewidth=0.5))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3,
          frameon=False, fontsize=7)
ax.set_xlim(0, 0.08)

plt.tight_layout()
save_fig('ch2_gain_loss_asymmetry')

## Aggregational Gaussianity

In [ ]:
# Compute weekly and monthly returns
weekly_ret = log_ret.resample('W').sum().dropna()
monthly_ret = log_ret.resample('ME').sum().dropna()

fig, ax = plt.subplots(figsize=(7, 3.5))

x = np.linspace(-5, 5, 300)

# Standardize each series and plot KDE
for ret, label, color, lw in [
    (log_ret, 'Daily', CRIMSON, 1.2),
    (weekly_ret, 'Weekly', AMBER, 1.2),
    (monthly_ret, 'Monthly', FOREST, 1.2),
]:
    z_ret = (ret - ret.mean()) / ret.std()
    kde = stats.gaussian_kde(z_ret)
    ax.plot(x, kde(x), color=color, linewidth=lw, label=label)

# Normal reference
ax.plot(x, stats.norm.pdf(x), color=MAIN_BLUE, linewidth=0.8, linestyle='--',
        label='Normal')

ax.set_xlabel('Standardized return')
ax.set_ylabel('Density')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4,
          frameon=False, fontsize=7)
ax.set_xlim(-5, 5)
ax.set_ylim(0, None)

# Annotate convergence
ax.annotate('Daily: heavy tails', xy=(-3.5, 0.01), fontsize=7, color=CRIMSON)
ax.annotate('Monthly: near-Normal', xy=(2.0, 0.35), fontsize=7, color=FOREST)

plt.tight_layout()
save_fig('ch2_aggregational_gaussianity')

# Summary statistics
print('\n   Excess kurtosis by horizon:')
print(f'   Daily:    {log_ret.kurtosis():.3f}')
print(f'   Weekly:   {weekly_ret.kurtosis():.3f}')
print(f'   Monthly:  {monthly_ret.kurtosis():.3f}')
print('   (Normal = 0, convergence toward 0 at longer horizons)')